# Mengukur Jarak Tipe Data Campuran (Gower Distance)

## Tujuan Perhitungan Jarak
Tujuan utama dari perhitungan jarak pada dataset ini adalah untuk mengukur tingkat kemiripan (*similarity*) atau ketidakmiripan (*dissimilarity*) antar pelanggan asuransi perjalanan. Analisis ini sangat berguna untuk:
1. **Segmentasi Pelanggan**: Mengelompokkan pelanggan yang memiliki profil risiko atau latar belakang ekonomi serupa.
2. **Sistem Rekomendasi**: Memberikan penawaran paket asuransi yang tepat berdasarkan kemiripan profil pelanggan baru dengan pelanggan lama.
3. **Analisis Karakteristik**: Memahami fitur dominan yang membedakan satu kelompok pelanggan dengan pelanggan lainnya.

## Data Understanding
Dataset **Travel Insurance Prediction** digunakan untuk menganalisis karakteristik pelanggan. Dataset ini memiliki tipe data campuran (Numerik dan Kategorikal).

### Deskripsi Atribut
| No | Atribut | Tipe Data | Alasan Penggunaan |
|---|---|---|---|
| 1 | Age | Numerik | Menunjukkan usia pelanggan untuk melihat pola kedekatan umur |
| 2 | Employment Type | Kategorikal | Membedakan sektor pekerjaan (Government/Private) |
| 3 | GraduateOrNot | Kategorikal | Menunjukkan status pendidikan terakhir pelanggan |
| 4 | AnnualIncome | Numerik | Mewakili tingkat pendapatan tahunan pelanggan |
| 5 | EverTravelledAbroad | Kategorikal | Menunjukkan pengalaman perjalanan internasional pelanggan |

### Missing Value dan Penanganan
Pengecekan *missing value* penting dilakukan karena metode perhitungan jarak tidak dapat memproses nilai kosong (NaN).

In [9]:
import pandas as pd
import numpy as np

# Load Dataset
df = pd.read_csv('TravelInsurancePrediction.csv')

# Seleksi fitur utama
selected_features = ['Age', 'Employment Type', 'GraduateOrNot', 'AnnualIncome', 'EverTravelledAbroad']
df_check = df[selected_features]

# Cek Missing Value
print("Hasil Pengecekan Missing Value per Kolom:")
print(df_check.isnull().sum())

# Menampilkan 5 data pertama untuk sampel
df_subset = df_check.head(5).copy()
print("\nSampel 5 Data Pertama:")
display(df_subset)

Hasil Pengecekan Missing Value per Kolom:
Age                    0
Employment Type        0
GraduateOrNot          0
AnnualIncome           0
EverTravelledAbroad    0
dtype: int64

Sampel 5 Data Pertama:


,Age,Employment Type,GraduateOrNot,AnnualIncome,EverTravelledAbroad
0,31,Government Sector,Yes,400000,No
1,31,Private Sector/Self Employed,Yes,1250000,No
2,34,Private Sector/Self Employed,Yes,500000,No
3,28,Private Sector/Self Employed,Yes,700000,No
4,28,Private Sector/Self Employed,Yes,700000,No


## Metode Perhitungan: Gower Distance

Gower Distance menghitung rata-rata dari perbedaan pada setiap fitur $k$ antara dua objek $i$ dan $j$:

$$d(i,j) = \frac{\sum_{k=1}^{n} s_{ij}^{(k)}}{n}$$

### Aturan Perhitungan Skor ($s_{ij}^{(k)}$):
1. **Fitur Numerik (Age, AnnualIncome):**
   Dihitung dengan membagi selisih absolut dengan rentang (range) nilai fitur tersebut:
   $$s_{ij}^{(k)} = \frac{|x_{ik} - x_{jk}|}{R_k}$$
2. **Fitur Kategorikal (Employment, Graduate, Abroad):**
   Bernilai **0** jika kategorinya sama, dan **1** jika berbeda.

### Contoh Perhitungan Manual (Pelanggan 1 vs Pelanggan 2)
Berdasarkan data sampel di atas:
* **Age:** $|31 - 31| / 10 = 0$
* **Employment Type:** Gov vs Private (Beda) = $1$
* **GraduateOrNot:** Yes vs Yes (Sama) = $0$
* **AnnualIncome:** $|400.000 - 1.250.000| / 1.500.000 = 0.5667$
* **EverTravelledAbroad:** No vs No (Sama) = $0$

**Total Jarak Gower:**
$$d(1,2) = \frac{0 + 1 + 0 + 0.5667 + 0}{5} = \mathbf{0.3133}$$

In [10]:
def calculate_gower(row1, row2, types, ranges):
    dist = 0
    for i, col in enumerate(row1.index):
        if types[i] == 'num':
            dist += abs(row1[col] - row2[col]) / ranges[col]
        else:
            dist += 1 if row1[col] != row2[col] else 0
    return dist / len(row1)

# Parameter (Range: Age=10, Income=1.500.000)
ranges = {'Age': 10, 'AnnualIncome': 1500000}
types = ['num', 'cat', 'cat', 'num', 'cat']

n = len(df_subset)
matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        matrix[i,j] = calculate_gower(df_subset.iloc[i], df_subset.iloc[j], types, ranges)

df_gower = pd.DataFrame(matrix, 
                        columns=['Pelanggan 1', 'Pelanggan 2', 'Pelanggan 3', 'Pelanggan 4', 'Pelanggan 5'], 
                        index=['Pelanggan 1', 'Pelanggan 2', 'Pelanggan 3', 'Pelanggan 4', 'Pelanggan 5'])

print("Matriks Jarak Gower (Output Python):")
display(df_gower)

Matriks Jarak Gower (Output Python):


,Pelanggan 1,Pelanggan 2,Pelanggan 3,Pelanggan 4,Pelanggan 5
Pelanggan 1,0.000000,0.313333,0.273333,0.300000,0.300000
Pelanggan 2,0.313333,0.000000,0.160000,0.133333,0.133333
Pelanggan 3,0.273333,0.160000,0.000000,0.146667,0.146667
Pelanggan 4,0.300000,0.133333,0.146667,0.000000,0.000000
Pelanggan 5,0.300000,0.133333,0.146667,0.000000,0.000000


### Interpretasi Hasil
1. **Nilai Diagonal (0.0000)**: Menunjukkan jarak objek terhadap dirinya sendiri adalah nol (simetris).
2. **Skala Jarak**: Semakin mendekati 0, kedua pelanggan semakin **mirip**. Semakin mendekati 1, kedua pelanggan semakin **berbeda**.
3. **Hasil Analisis**: Pelanggan 1 dan Pelanggan 2 memiliki jarak 0.3133, yang menunjukkan ketidakmiripan moderat yang dipengaruhi oleh perbedaan sektor pekerjaan dan pendapatan tahunan.

## Implementasi Menggunakan Orange Data Mining
Berikut adalah validasi alur kerja menggunakan Orange Data Mining:

![Workflow Orange](workflow_campuran.png)


![Gower Matrix](gower_matrix.png)